In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🧪 W7-D1 概念实验：数字员工的行为由什么决定？

> 配套阅读：`第7周-Day1-数字员工总览与Agent行为设计.md`（概念讲解、案例与术语表在那边）
> 本 notebook 只回答一个问题：**「数字员工」不是一个 prompt 字符串，而是 角色 + 权限 + 工具 + 护栏 + 输出契约 的组合体** —— 用 4 个小实验验证每一层配置如何真实地改变行为。
>
> 实验环境：纯 Python 标准库 + numpy/matplotlib 模拟，不调用任何 LLM。


## 实验 1：同一个请求引擎，两份「员工配置」，行为差异从哪来？

数字员工的本质是一份**可执行的配置**：角色只是岗位描述，真正决定行为的是权限集合、可用工具和风险策略。
构建 `DigitalEmployee` dataclass，让两个员工（低权限客服 / 高权限运维）面对同一批请求，观察决策分叉。


## 实验 2：输出格式 = 交付标准 —— 无契约 vs JSON 契约的机器可解析率

企业里员工交付有模板，数字员工的交付标准就是**输出契约**。
模拟 50 条自由文本回答 vs 50 条走统一 JSON schema 的回答，统计下游程序（`json.loads` + 字段校验）的解析成功率。


## 实验 3：多轮对话的上下文裁剪 —— 固定 token 预算下，哪种策略关键信息保留率最高？

上下文窗口有限、历史越长噪音越多。常见裁剪策略：① 硬截断（只留最近）② 系统规则+滑窗 ③ 混合（系统永不丢 + 关键消息优先 + 尾部滑窗）。
构造一段 40 轮对话，埋入 5 个关键事实（订单号、退货诉求等），看预算从 200→2000 token 时各策略的保真度。


## 实验 4：护栏矩阵 —— 风险等级 × 员工权限等级如何映射到最终动作？

护栏不是一句话，是一张**决策矩阵**：风险越高、权限越低，动作越保守。
把矩阵显式化，并用 200 条随机请求统计分流比例（执行/确认/升级人工/拒绝）。


## 结论

| 配置层 | 实验验证的行为差异 |
|---|---|
| 角色+权限+工具 | 同一请求在两个员工处得到不同决策（实验1） |
| 输出契约 | JSON 契约解析率 100% vs 自由文本 0%（实验2） |
| 上下文策略 | 混合裁剪在低预算下保真度最高（实验3） |
| 护栏矩阵 | 风险×权限显式映射，可单测（实验4） |

**数字员工 = 把「岗位说明书」编译成可执行的决策逻辑。**
→ 深入阅读：同名 `.md` 的 System Prompt 四层结构、SOUL.md 与 OpenClaw vs Hermes 对比。
